In [ ]:
from lanchain_google_vertexai import ChatVertexAI
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from typing import TypedDict, List
from langgraph import StateGraph, START, END
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
llm = ChatVertexAI(modelname = 'gemini-pro')

In [ ]:
documents = "Get the documents from the db via api call"
embeddings = HuggingFaceEndpointEmbeddings(
    repo_id="LiquidAI/LFM2.5-Embedding-350M",
    task="feature-extraction" 
)
chunks = " "
retriver = FAISS.from_documents(chunks , embeddings)

In [ ]:
class State(TypedDict):
    question: str
    answer: str
    docs: List[Document]

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
    ("system", "Answer only from the context, if not in context, say you don't know."),
    ("human","Question: {question}\n\nContext:\n{context}"),
    ]
)

In [ ]:
def retrive(state):
    q = state['question']
    return {"docs": retriver.invoke(q)}

In [ ]:
def generate(state):
    context = "\n\n".join(d.page_content for d in state['docs'])
    out = (prompt | llm).invoke({"question": state['question'], "context": context})
    return {"answer": out.content}

In [ ]:
graph = StateGraph(State)
graph.add_node("retrive",retrive)
graph.add_node("generate",generate)

graph.add_edge(START,"retrive")
graph.add_edge("retrive","generate")
graph.add_edge("generate",END)
app = graph.compile()